# Deploy Docker Containers
 

## Import the FABlib Library

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

## Create the Experiment Slice

Start a docker container with docker compose and post boot tasks.

In [ ]:
# Create a slice
slice = fablib.new_slice(name="MyDocker")

# Add a node
node = slice.add_node(name="Node1", disk=10, image='docker_rocky_8')

node.add_post_boot_upload_directory('node_tools','.')
node.add_post_boot_execute('node_tools/enable_docker.sh {{ _self_.image }} ')
node.add_post_boot_upload_directory('docker_containers','.')
node.add_post_boot_execute('cd docker_containers/fabric_multitool ; docker compose up -d ')

# Submit the slice
slice.submit();

## Run a Docker Container from a Command Line

In [ ]:
slice.update()

print("Slice state:", slice.get_state())
print("Slice stable:", slice.isStable())

for node in slice.get_nodes():
    print("----", node.get_name(), "----")
    print("reservation state:", node.get_reservation_state())
    print("management ip:", node.get_management_ip())
    print("username:", node.get_username())
    print("error:", node.get_error_message())
    print(node.get_ssh_command())

In [ ]:
node = slice.get_node('Node1')

stdout, stderr = node.execute("docker run -d -it "
                                "--name fabric_command_line "
                                "fabrictestbed/slice-vm-rocky8-multitool:0.0.1 "
                                , output_file=f"{node.get_name()}.log");


## View your Containers

In [ ]:
stdout, stderr = node.execute('docker ps -a')

## Execute a Command in a Container

In [ ]:
stdout, stderr = node.execute('docker exec fabric_command_line ip addr list')

## Delete the Slice

Please delete your slice when you are done with your experiment.

In [ ]:
slice.delete()